# 商品明细表清洗：第一步，检查粒度和组合主键

本次只做读取、认识字段和主键检查，保留原始值。后面再进行金额转换、运费检查和关联核对。

一行代表某个订单中的一条商品明细。`order_id` 识别订单，`order_item_id` 是订单内的明细序号，可以在不同订单中重复。两列共同识别一条明细。

| 字段 | 含义 |
| --- | --- |
| order_id | 订单 ID，关联订单表 |
| order_item_id | 该订单内的商品明细序号 |
| product_id | 商品 ID，关联商品表 |
| seller_id | 卖家 ID，关联卖家表 |
| shipping_limit_date | 卖家交物流的截止时间 |
| price | 该条商品明细的商品金额 |
| freight_value | 该条商品明细记录的运费 |

金额单位为 BRL；此时先保留字符串，后续明确转换规则。

In [1]:
from pathlib import Path
import pandas as pd

project_dir = Path(r'D:\数据分析\olist-operations-analytics')
data_dir = project_dir.parent / 'archive'

items_raw = pd.read_csv(
    data_dir / 'olist_order_items_dataset.csv',
    dtype=str, encoding='utf-8-sig',
    keep_default_na=False, na_values=['']
)
print('行数和列数：', items_raw.shape)
items_raw.head(3)

行数和列数： (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87


In [2]:
items_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   order_id             112650 non-null  object
 1   order_item_id        112650 non-null  object
 2   product_id           112650 non-null  object
 3   seller_id            112650 non-null  object
 4   shipping_limit_date  112650 non-null  object
 5   price                112650 non-null  object
 6   freight_value        112650 non-null  object
dtypes: object(7)
memory usage: 6.0+ MB


## 检查组合主键

`duplicated(subset=key_cols)` 只比较指定的两列：两列都与某条此前记录相同时，才标记为重复。

例如 `(订单 A, 序号 1)` 和 `(订单 B, 序号 1)` 是不同明细；`(订单 A, 序号 1)` 与 `(订单 A, 序号 2)` 也不同。

主键检查需同时查看缺失与唯一性。重复数量默认只统计首次出现后的记录。

In [3]:
key_cols = ['order_id', 'order_item_id']

item_key_check = pd.Series({
    '商品明细行数': len(items_raw),
    '订单ID不同值数': items_raw['order_id'].nunique(),
    '商品明细序号不同值数': items_raw['order_item_id'].nunique(),
    '订单ID缺失行数': items_raw['order_id'].isna().sum(),
    '商品明细序号缺失行数': items_raw['order_item_id'].isna().sum(),
    '组合主键重复行数_首条之后': items_raw.duplicated(subset=key_cols).sum(),
    '整行重复数_首条之后': items_raw.duplicated().sum()
})

item_key_check

商品明细行数           112650
订单ID不同值数          98666
商品明细序号不同值数           21
订单ID缺失行数              0
商品明细序号缺失行数            0
组合主键重复行数_首条之后         0
整行重复数_首条之后            0
dtype: int64

In [4]:
# 找一笔恰好包含两条明细的订单，观察订单内的序号。
order_item_counts = items_raw['order_id'].value_counts()
example_order = order_item_counts[order_item_counts.eq(2)].index[0]

items_raw.loc[
    items_raw['order_id'].eq(example_order),
    ['order_id', 'order_item_id', 'product_id', 'price', 'freight_value']
]

,order_id,order_item_id,product_id,price,freight_value
88714,c97de78b8ee3343c72b234c57b872408,1,1e1bd97639fea164c61d7f2260cb4033,199.99,82.84
88715,c97de78b8ee3343c72b234c57b872408,2,1e1bd97639fea164c61d7f2260cb4033,199.99,82.84


## 我的观察

1. 商品明细有多少行，涉及多少个订单？

我的回答：商品明细行数有112650行，98666个订单

2. 为什么 `order_id` 和 `order_item_id` 都不能单独作为本表主键？

一个订单有多条明细，所以 order_id 会重复；不同订单会重复使用相同的明细序号，所以 order_item_id 也会重复。两列组合才能唯一识别一条明细。

3. 组合主键是否非空且唯一？这一步是否需要去重？

我的回答：组合主键非空且唯一，这一步无需去重。

核对提示：本机原始表为 112,650 行，涉及 98,666 个订单，两列组合键无缺失和重复。订单覆盖差异留到关联检查时结合状态解释。

下一小步：将 `price`、`freight_value` 转为数值，检查无法解析、负数与零值，再学习按订单汇总。

In [5]:
key_cols = ['order_id', 'order_item_id']

print(items_raw[key_cols].isna().sum())

print(
    '组合主键重复数：',
    items_raw.duplicated(subset=key_cols).sum()
)

order_id         0
order_item_id    0
dtype: int64
组合主键重复数： 0


In [6]:
items_clean = items_raw.copy()

money_cols = ['price', 'freight_value']
parse_failures = {}

for col in money_cols:
    items_clean[col] = pd.to_numeric(
        items_raw[col],
        errors='coerce'
    )

    parse_failures[col] = (
        items_raw[col].notna()
        & items_clean[col].isna()
    ).sum()

In [7]:
money_check = pd.DataFrame({
    '转换后类型': items_clean[money_cols].dtypes.astype(str),
    '原始缺失数': items_raw[money_cols].isna().sum(),
    '解析失败数': pd.Series(parse_failures),
    '负数记录数': items_clean[money_cols].lt(0).sum(),
    '零值记录数': items_clean[money_cols].eq(0).sum(),
    '最小值': items_clean[money_cols].min(),
    '最大值': items_clean[money_cols].max()
})

money_check

,转换后类型,原始缺失数,解析失败数,负数记录数,零值记录数,最小值,最大值
price,float64,0,0,0,0,0.85,6735.00
freight_value,float64,0,0,0,383,0.00,409.68


In [8]:
from decimal import Decimal

def to_cents(raw_value):
    amount = Decimal(raw_value)

    if not amount.is_finite():
        raise ValueError('金额必须是有限数值')

    cents = amount * 100

    if cents != cents.to_integral_value():
        raise ValueError('金额无法精确转换为整数分，需要核查')

    return int(cents)

In [9]:
items_clean['price_cents'] = (
    items_raw['price'].map(to_cents)
)

items_clean['freight_cents'] = (
    items_raw['freight_value'].map(to_cents)
)

# 业务特征标记，不代表数据错误
items_clean['is_zero_freight'] = (
    items_clean['freight_cents'].eq(0)
)

In [10]:
cent_cols = ['price_cents', 'freight_cents']

print(items_clean[cent_cols].dtypes)
print('零运费明细数：', items_clean['is_zero_freight'].sum())

items_clean[cent_cols].agg(['min', 'max'])

price_cents      int64
freight_cents    int64
dtype: object
零运费明细数： 383


,price_cents,freight_cents
min,85,0
max,673500,40968


In [16]:
order_item_summary = (
    items_clean
    .groupby('order_id', as_index=False)
    .agg(
        item_line_count=('order_item_id', 'size'),
        product_count=('product_id', 'nunique'),
        seller_count=('seller_id', 'nunique'),
        goods_amount_cents=('price_cents', 'sum'),
        freight_amount_cents=('freight_cents', 'sum')
    )
)

print('汇总后的订单数：', len(order_item_summary))
order_item_summary.head()

汇总后的订单数： 98666


,order_id,item_line_count,product_count,seller_count,goods_amount_cents,freight_amount_cents
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,5890,1329
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,23990,1993
2,000229ec398224ef6ca0657da4fc703e,1,1,1,19900,1787
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,1299,1279
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,19990,1814


In [12]:
reconcile = pd.DataFrame({
    '汇总前': [
        len(items_clean),
        items_clean['price_cents'].sum(),
        items_clean['freight_cents'].sum()
    ],
    '汇总后': [
        order_item_summary['item_line_count'].sum(),
        order_item_summary['goods_amount_cents'].sum(),
        order_item_summary['freight_amount_cents'].sum()
    ]
}, index=['商品明细数量', '商品金额（分）', '运费（分）'])

reconcile['差额'] = reconcile['汇总后'] - reconcile['汇总前']

reconcile

,汇总前,汇总后,差额
商品明细数量,112650,112650,0
商品金额（分）,1359164370,1359164370,0
运费（分）,225190954,225190954,0


In [13]:
zero_freight_orders = (
    order_item_summary['freight_amount_cents'].eq(0).sum()
)

print('零运费订单数：', zero_freight_orders)

零运费订单数： 338


In [17]:
orders_base = pd.read_csv(
    data_dir / 'olist_orders_dataset.csv',
    usecols=['order_id', 'order_status'],
    dtype=str,
    encoding='utf-8-sig',
    keep_default_na=False,
    na_values=['']
)

orphan_summary = order_item_summary.loc[
    ~order_item_summary['order_id'].isin(orders_base['order_id'])
]

print('商品汇总中找不到主表的订单数：', len(orphan_summary))

if not orphan_summary.empty:
    raise ValueError('存在找不到订单主表的记录，请先核查')

商品汇总中找不到主表的订单数： 0


In [18]:
orders_with_items = orders_base.merge(
    order_item_summary,
    on='order_id',
    how='left',
    validate='one_to_one',
    indicator='items_match'
)

In [20]:
summary_cols = [
    'item_line_count',
    'product_count',
    'seller_count',
    'goods_amount_cents',
    'freight_amount_cents'
]

for col in summary_cols:
    orders_with_items[col] = orders_with_items[col].astype('Int64')

In [21]:
print('关联后订单行数：', len(orders_with_items))

print(orders_with_items['items_match'].value_counts())

关联后订单行数： 99441
items_match
both          98666
left_only       775
right_only        0
Name: count, dtype: int64


In [22]:
orders_with_items.loc[
    orders_with_items['items_match'].eq('left_only'),
    'order_status'
].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [23]:
orders_with_items['flag_progressed_missing_items'] = (
    orders_with_items['items_match'].eq('left_only')
    & orders_with_items['order_status'].isin(
        ['approved', 'invoiced', 'processing', 'shipped', 'delivered']
    )
)

print(
    '优先核查订单数：',
    orders_with_items['flag_progressed_missing_items'].sum()
)

优先核查订单数： 3


In [24]:
delivered = orders_with_items.loc[
    orders_with_items['order_status'].eq('delivered')
].copy()

# 求和前检查金额完整性，避免缺失值被求和操作跳过
amount_cols = ['goods_amount_cents', 'freight_amount_cents']

if delivered[amount_cols].isna().any().any():
    raise ValueError('已交付订单存在金额缺失，请先核查')

delivered_order_count = len(delivered)
gmv_cents = int(delivered['goods_amount_cents'].sum())
freight_cents = int(delivered['freight_amount_cents'].sum())

print('已交付订单数：', delivered_order_count)
print('已交付商品成交额：{:,.2f} BRL'.format(gmv_cents / 100))
print('已交付运费：{:,.2f} BRL'.format(freight_cents / 100))
print(
    '商品客单价：{:,.2f} BRL'.format(
        gmv_cents / delivered_order_count / 100
    )
)

已交付订单数： 96478
已交付商品成交额：13,221,498.11 BRL
已交付运费：2,198,275.64 BRL
商品客单价：137.04 BRL


In [25]:
items_clean['shipping_limit_date'] = pd.to_datetime(
    items_raw['shipping_limit_date'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

print('原始缺失数：', items_raw['shipping_limit_date'].isna().sum())

print('非空但解析失败数：', (
    items_raw['shipping_limit_date'].notna()
    & items_clean['shipping_limit_date'].isna()
).sum())

print('转换后类型：', items_clean['shipping_limit_date'].dtype)

原始缺失数： 0
非空但解析失败数： 0
转换后类型： datetime64[ns]


In [26]:
product_keys = pd.read_csv(
    data_dir / 'olist_products_dataset.csv',
    usecols=['product_id'],
    dtype=str
)['product_id']

seller_keys = pd.read_csv(
    data_dir / 'olist_sellers_dataset.csv',
    usecols=['seller_id'],
    dtype=str
)['seller_id']

In [27]:
def check_reference(detail_key, reference_key):
    return {
        '维表主键缺失数': reference_key.isna().sum(),
        '维表主键重复数': reference_key.duplicated().sum(),
        '明细外键缺失数': detail_key.isna().sum(),
        '非空但未匹配明细数': (
            detail_key.notna()
            & ~detail_key.isin(reference_key)
        ).sum()
    }

reference_check = pd.DataFrame({
    '商品关联': check_reference(
        items_clean['product_id'], product_keys
    ),
    '卖家关联': check_reference(
        items_clean['seller_id'], seller_keys
    )
})

reference_check

,商品关联,卖家关联
维表主键缺失数,0,0
维表主键重复数,0,0
明细外键缺失数,0,0
非空但未匹配明细数,0,0
